# Lab 5 – Classes: your own types of objects
**Lecture 5:** classes · attributes and methods · "function overloading" · inheritance · polymorphism
**Project module today:** `src/models.py`

| Part | What you do | Data |
|---|---|---|
| 1 | Setup: connects to your project and checks Lab 2 + Lab 4 | your project |
| 2 | Classes step by step (lecture 5) | 🧪 TEST DATA (Pokémon) |
| 3 | Three kinds of project classes: `Dataset`, the `Analysis` family, the `Filter` family | 🧪 TEST DATA |
| 4 | Put the classes into `src/models.py` and use them in `main.ipynb` | **your data** |
| 5 | Result check → `docs/lab05_check.md` | your project |

**How to work**
- Run the cells from top to bottom (▶ on the left, or Shift+Enter).
- `...` = a place for **your** code. Replace every `...`.
- In every code cell the comments **# Step 1, Step 2, …** tell you exactly what to write, line by line. Do them in order and compare with the *expected* result in the comment.
- Every task has the same order: short explanation → example on something else (songs, tickets – not Pokémon) → **your task** (you must change and adapt the idea, copying the example is not enough) → **✔️ Check** cell.
- 💡 **Hint** = click to open, only when you are stuck. 📌 = the exact place in the Python documentation.
- This notebook **does not change anything** in your project. You write and paste code into your files yourself.
- 🧪 **TEST DATA = Pokémon. It is NOT your data** – only for practice. Your own data comes in part 4.

🏆 **For those who finish today:** show the teacher the **✅ 6/6** at the end of part 5 before you leave.

## 1. Setup – run this first
The cell below connects to Google Drive (click **Allow**), finds your project `PyDS_project` and checks whether **Lab 2** (`loader.py` reads your data) and **Lab 4** (`stats.py` + `main.ipynb`) are done. It **only reads** – it does not create or change any file.

- **✅ everything OK** → you can do parts 2–5.
- **❌ a numbered list** → each point says **which lab and which step** fixes it. Fix the points in this order, then run the Setup again.
  Meanwhile you can already work on **parts 2 and 3** – they use the 🧪 TEST DATA, not your project. Parts 4–5 wait until the list is empty.

In [1]:
# @title ⚙️ Setup Lab 5 – run this cell first (click ▶ on the left)
# This cell only READS: it connects to Google Drive, finds your project and checks
# Lab 2 (loader.py) and Lab 4 (stats.py + main.ipynb). It creates or changes NOTHING
# in your project. It also loads the 🧪 TEST DATA (Pokémon) into memory.
import sys, os, json, csv, io, re, importlib, contextlib, datetime, types, urllib.request
from pathlib import Path
sys.dont_write_bytecode = True          # do not leave __pycache__ files in your project

PROJECT_NAME = "PyDS_project"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_NAME
except ImportError:                     # VS Code / PyCharm / Jupyter on your computer
    IN_COLAB = False
    PROJECT = Path.cwd() / PROJECT_NAME
    for _folder in [Path.cwd(), *Path.cwd().parents]:
        if (_folder / "src" / "loader.py").exists():
            PROJECT = _folder
            break

# ---------------------------------------------------------------- 🧪 TEST DATA
TEST_URL = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"

def _test_text():
    for p in [Path("pokemon_data.csv"), Path.cwd().parent / "pokemon_data.csv"]:
        if p.exists():
            return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(TEST_URL, timeout=30) as r:
        return r.read().decode("utf-8")

def _int(text):
    text = (text or "").strip()
    return int(text) if text else None

TEST = []
try:
    for _r in csv.DictReader(io.StringIO(_test_text())):
        TEST.append({
            "Name": _r["Name"], "Type 1": _r["Type 1"], "Type 2": _r["Type 2"] or None,
            "HP": _int(_r["HP"]), "Attack": _int(_r["Attack"]), "Defense": _int(_r["Defense"]),
            "Speed": _int(_r["Speed"]), "Generation": _int(_r["Generation"]),
            "Legendary": _r["Legendary"].strip().lower() == "true",
        })
except Exception as e:
    print("❌ Could not load the TEST DATA:", e, "\n   Check the internet connection and run this cell again.")

# ---------------------------------------------------------------- helpers
def fresh(module):
    """Import the NEWEST version of src/<module>.py from YOUR project."""
    for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
        del sys.modules[name]
    if str(PROJECT) in sys.path:
        sys.path.remove(str(PROJECT))
    sys.path.insert(0, str(PROJECT))
    importlib.invalidate_caches()
    return importlib.import_module("src." + module)

class ProjectNotReady(Exception):
    pass

def require_project():
    if not PROJECT_OK:
        raise ProjectNotReady(
            "⛔ This part works on YOUR project, and your project is not ready yet.\n"
            "Fix these points first:\n  • " + "\n  • ".join(PROBLEMS) +
            "\nThen run the Setup cell (part 1) again. Parts 2 and 3 work anyway (TEST DATA).")

RESULTS = {}

def check(label, test, hint="", details=None):
    try:
        ok, err = bool(test()), None
    except Exception as e:
        ok, err = False, e
    RESULTS[label] = ok
    if ok:
        print("✅", label)
        return True
    print("❌", label)
    for d in details or []:
        print("   •", d)
    if hint:
        print("   →", hint)
    if isinstance(err, NameError):
        print("   (Python says:", str(err) + " → did you run your cell above with ▶? Is the name spelled exactly like this?)")
    elif err is not None:
        print("   (Python says:", type(err).__name__ + ":", str(err) + ")")
    return False

def summary(prefix):
    labels = [l for l in RESULTS if l.startswith(prefix)]
    good = sum(RESULTS[l] for l in labels)
    print(f"\n{good}/{len(labels)} OK" + ("  🎉" if good == len(labels) else "  → fix the ❌ above and run this check again"))

def safe_test(test, *args):
    """Run one of the test_... functions; an error in YOUR code becomes one readable point."""
    try:
        return test(*args)
    except Exception as e:
        tip = " → there is still a ... in your class" if "ellipsis" in str(e) else ""
        return [f"your code stops with {type(e).__name__}: {e}{tip}"]

def close(a, b):
    return a is not None and b is not None and abs(a - b) < 0.011

def run_main():
    """Run the code cells of YOUR main.ipynb from top to bottom (like Run all). Returns (ok, message, namespace)."""
    nb = json.loads((PROJECT / "main.ipynb").read_text(encoding="utf-8"))
    cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
    ns = {"__name__": "__main__"}
    try:
        ns["get_ipython"] = get_ipython
    except NameError:
        pass
    try:
        from IPython.core.inputtransformer2 import TransformerManager
        transform = TransformerManager().transform_cell
    except Exception:
        transform = lambda s: s
    old_cwd, old_path = os.getcwd(), list(sys.path)
    ok, message = True, f"{len(cells)} code cells ran without errors"
    try:
        os.chdir(PROJECT)
        for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
            del sys.modules[name]
        importlib.invalidate_caches()
        for i, c in enumerate(cells, 1):
            src = "".join(c.get("source", []))
            if not src.strip():
                continue
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    exec(compile(transform(src), f"main.ipynb code cell {i}", "exec"), ns)
            except Exception as e:
                first = src.strip().splitlines()[0][:60]
                ok, message = False, f"code cell {i} (starts with: {first}) → {type(e).__name__}: {e}"
                break
    finally:
        os.chdir(old_cwd)
        sys.path[:] = old_path
        for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
            del sys.modules[name]
    return ok, message, ns

def main_code():
    nb = json.loads((PROJECT / "main.ipynb").read_text(encoding="utf-8"))
    if not nb.get("cells"):
        raise ValueError("main.ipynb has no cells")
    return "\n".join("".join(c.get("source", [])) for c in nb.get("cells", []) if c.get("cell_type") == "code")

# ---------- tests of the part 3 classes (used for the notebook AND for your src/models.py)
def test_dataset(Dataset):
    bad = []
    d = Dataset(TEST, "pokemon")
    if len(d) != 800: bad.append("len(Dataset(TEST)) should be 800 → __len__ returns len(self.rows)")
    if str(d) != "Dataset 'pokemon': 800 rows": bad.append(f"str(...) gives {str(d)!r}, should be \"Dataset 'pokemon': 800 rows\"")
    if len(d.column("Type 2")) != 414: bad.append("column('Type 2') should have 414 values (None skipped)")
    w = d.where("Type 1", "Water")
    if type(w) is not Dataset or len(w) != 112: bad.append("where('Type 1', 'Water') should return a NEW Dataset with 112 rows")
    elif w.name != "pokemon | Type 1 = Water": bad.append(f"name of the where(...) result is {w.name!r}, should be 'pokemon | Type 1 = Water'")
    elif len(w.where("Generation", 1)) != 31: bad.append("where(...).where('Generation', 1) should give 31 rows")
    if (d.count_by("Type 1") or {}).get("Water") != 112: bad.append("count_by('Type 1')['Water'] should be 112")
    if not close((d.group_stat("Type 1", "Attack") or {}).get("Dragon"), 112.12): bad.append("group_stat('Type 1', 'Attack')['Dragon'] should be 112.12")
    return bad

def test_analyses(ns):
    bad = []
    for name in ["Analysis", "CountBy", "GroupMean", "TopGroups"]:
        if not hasattr(ns, name):
            return [f"class {name} is missing"]
    A, C, G, T = ns.Analysis, ns.CountBy, ns.GroupMean, ns.TopGroups
    if not (issubclass(C, A) and issubclass(G, A)): bad.append("CountBy and GroupMean must be children of Analysis: class CountBy(Analysis):")
    if not issubclass(T, G): bad.append("TopGroups must be a child of GroupMean: class TopGroups(GroupMean):")
    d = ns.Dataset(TEST, "pokemon")
    c = C("Type 1")
    if str(c) != "Analysis: count by Type 1": bad.append(f"str(CountBy('Type 1')) gives {str(c)!r}, should be 'Analysis: count by Type 1'")
    if (c.run(d) or {}).get("Water") != 112: bad.append("CountBy('Type 1').run(data)['Water'] should be 112")
    g = G("Type 1", "Attack")
    if g.name != "mean Attack by Type 1": bad.append(f"GroupMean name is {g.name!r}, should be 'mean Attack by Type 1'")
    if not close((g.run(d) or {}).get("Dragon"), 112.12): bad.append("GroupMean('Type 1', 'Attack').run(data)['Dragon'] should be 112.12")
    t = T("Type 1", "Attack", 3)
    if t.name != "top 3 Type 1 by mean Attack": bad.append(f"TopGroups name is {t.name!r}, should be 'top 3 Type 1 by mean Attack'")
    r = t.run(d)
    if not (isinstance(r, list) and len(r) == 3 and r[0][0] == "Dragon" and r[2][0] == "Ground"): bad.append(f"TopGroups('Type 1', 'Attack', 3).run(data) gives {r!r}, should start with ('Dragon', 112.12) and have 3 pairs")
    if len(T("Type 1", "Attack", 2).run(d) or []) != 2: bad.append("TopGroups(..., 2) should give 2 pairs")
    if not str(t.report(d)).startswith("top 3 Type 1 by mean Attack: [('Dragon', 112.12)"): bad.append("report() of TopGroups should start with \"top 3 Type 1 by mean Attack: [('Dragon', 112.12)\"")
    return bad

def test_filters(ns):
    bad = []
    for name in ["Filter", "Equals", "Above"]:
        if not hasattr(ns, name):
            return [f"class {name} is missing"]
    F, E, Ab = ns.Filter, ns.Equals, ns.Above
    if not (issubclass(E, F) and issubclass(Ab, F)): bad.append("Equals and Above must be children of Filter")
    d = ns.Dataset(TEST, "pokemon")
    if str(F("HP")) != "filter on HP": bad.append(f"str(Filter('HP')) gives {str(F('HP'))!r}, should be 'filter on HP'")
    if F("HP").keep(TEST[0]) is not True: bad.append("Filter.keep(row) of the base class should return True")
    allrows = F("HP").apply(d)
    if type(allrows) is not ns.Dataset or len(allrows) != 800: bad.append("Filter('HP').apply(data) should return a new Dataset with all 800 rows")
    e = E("Type 1", "Water")
    if str(e) != "Type 1 = Water": bad.append(f"str(Equals('Type 1', 'Water')) gives {str(e)!r}, should be 'Type 1 = Water'")
    if e.keep({"Type 1": "Water"}) is not True or e.keep({"Type 1": "Fire"}) is not False: bad.append("Equals.keep(row) should be True only when row[column] == value")
    a = Ab("Attack", 100)
    if str(a) != "Attack > 100": bad.append(f"str(Above('Attack', 100)) gives {str(a)!r}, should be 'Attack > 100'")
    if a.keep({"Attack": None}) is not False or a.keep({"Attack": 101}) is not True or a.keep({"Attack": 100}) is not False:
        bad.append("Above.keep(row): None → False, 101 → True, 100 → False (strictly bigger)")
    strong = a.apply(d)
    if len(strong) != 170: bad.append(f"Above('Attack', 100).apply(data) should have 170 rows, has {len(strong)}")
    elif strong.name != "pokemon | Attack > 100": bad.append(f"name of the result is {strong.name!r}, should be 'pokemon | Attack > 100'")
    elif len(e.apply(strong)) != 17: bad.append("Equals('Type 1', 'Water').apply(strong) should have 17 rows")
    return bad

# ---------------------------------------------------------------- reference stats (only if yours is not ready)
def _ref_mean(values, digits=2):
    clean = [v for v in values if v is not None]
    return round(sum(clean) / len(clean), digits) if clean else None

def _ref_count_by(rows, key):
    counts = {}
    for row in rows:
        counts[row[key]] = counts.get(row[key], 0) + 1
    return counts

def _ref_group_stat(rows, group, value, how="mean"):
    groups = {}
    for row in rows:
        if row[value] is not None:
            groups.setdefault(row[group], []).append(row[value])
    if how == "max":
        return {k: max(v) for k, v in groups.items()}
    if how == "count":
        return {k: len(v) for k, v in groups.items()}
    return {k: _ref_mean(v) for k, v in groups.items()}

# ---------------------------------------------------------------- check YOUR project
PROBLEMS, NOTES = [], []
MY_ROWS, MY_TYPES, STATS_OK = None, {}, False
MINI = [{"name": "Bulbasaur", "type1": "Grass", "attack": 49}, {"name": "Ivysaur", "type1": "Grass", "attack": 62},
        {"name": "Charmander", "type1": "Fire", "attack": 52}, {"name": "Charmeleon", "type1": "Fire", "attack": 64},
        {"name": "Squirtle", "type1": "Water", "attack": 48}]

def _problem(where, text):
    PROBLEMS.append(f"[{where}] {text}")

if not PROJECT.exists():
    _problem("Lab 2, Setup", f"No project folder {PROJECT}. Open Lab 2 and run its Setup cell – it creates the project PyDS_project.")
elif not (PROJECT / "src" / "loader.py").exists():
    _problem("Lab 2, Setup", "src/loader.py is missing. Open Lab 2 and run its Setup cell – it creates the missing files.")
else:
    # ---- Lab 2: loader.py
    try:
        _L = fresh("loader")
        _name = getattr(_L, "DATA_FILE_NAME", "")
        _raw = PROJECT / "data" / "raw"
        _files = sorted(p.name for p in _raw.glob("*") if p.name != "README.md") if _raw.exists() else []
        if _name == "my_data.csv":
            _problem("Lab 2, step 3.3", 'src/loader.py, PART B: DATA_FILE_NAME is still "my_data.csv" → write the name of YOUR file.')
        elif str(_name).startswith("practice_"):
            _problem("Lab 2, step 3.1", f'DATA_FILE_NAME = "{_name}" is the practice file → upload YOUR data to data/raw/ and write its name.')
        elif not (_raw / str(_name)).exists():
            _problem("Lab 2, step 3.1", f'DATA_FILE_NAME = "{_name}", but there is no such file in data/raw/. Files there: {_files}. Upload it or correct the name (capital letters, .csv).')
        elif not getattr(_L, "COLUMN_TYPES", {}):
            _problem("Lab 2, step 3.3", "src/loader.py, PART B: COLUMN_TYPES is empty → write your columns and their types.")
        else:
            try:
                MY_ROWS = _L.load_project_data()
                MY_TYPES = dict(_L.COLUMN_TYPES)
                if not MY_ROWS:
                    _problem("Lab 2, step 3.4", "load_project_data() returned no rows → check your CSV file.")
            except Exception as e:
                _problem("Lab 2, step 3.4", f"load_project_data() does not work: {type(e).__name__}: {e}")
    except Exception as e:
        _problem("Lab 2", f"src/loader.py has an error and cannot be imported: {type(e).__name__}: {e}")

    # ---- Lab 4: stats.py
    if not (PROJECT / "src" / "stats.py").exists():
        _problem("Lab 4, step 4.6", "src/stats.py is missing → paste your Lab 4 functions into src/stats.py.")
    else:
        try:
            _S = fresh("stats")
            _tests = {
                "mean": (lambda f: f([1, 2, 4]), 2.33, "mean([1, 2, 4])"),
                "median": (lambda f: (f([3, 1, 2]), f([4, 1, 3, 2])), (2, 2.5), "median([3, 1, 2]), median([4, 1, 3, 2])"),
                "count_by": (lambda f: f(MINI, "type1"), {"Grass": 2, "Fire": 2, "Water": 1}, "count_by(MINI, 'type1')"),
                "group_stat": (lambda f: f(MINI, "type1", "attack"), {"Grass": 55.5, "Fire": 58.0, "Water": 48.0},
                               "group_stat(MINI, 'type1', 'attack')"),
            }
            _missing = [f"{fn}()" for fn in _tests if not hasattr(_S, fn)]
            _stats_bad = len(_missing)
            if _missing:
                _problem("Lab 4, part 4", f"src/stats.py has no {', '.join(_missing)} → write them (Lab 4, 4.1–4.4) and paste them into src/stats.py (4.6).")
            for _fn, (_call, _exp, _txt) in _tests.items():
                if not hasattr(_S, _fn):
                    continue
                try:
                    _got = _call(getattr(_S, _fn))
                except Exception as e:
                    _got = f"{type(e).__name__}: {e}"
                if _got != _exp:
                    _problem("Lab 4, part 4", f"{_fn}() in src/stats.py gives a wrong result: {_txt} → {_got!r}, expected {_exp!r}.")
                    _stats_bad += 1
            if _stats_bad == 0:
                STATS_OK = True
                mean, count_by, group_stat = _S.mean, _S.count_by, _S.group_stat
        except Exception as e:
            _problem("Lab 4, step 4.6", f"src/stats.py has an error and cannot be imported: {type(e).__name__}: {e}")

    # ---- models.py (today's file)
    if not (PROJECT / "src" / "models.py").exists():
        _problem("Lab 2, Setup", "src/models.py is missing → run the Setup cell of Lab 2 again (it only adds missing files).")
    elif "class " in (PROJECT / "src" / "models.py").read_text(encoding="utf-8"):
        NOTES.append("src/models.py already has some classes – you will add today's classes below them (part 4).")

    # ---- Lab 4: main.ipynb
    try:
        _code = main_code()
        if "src.stats" not in _code:
            _problem("Lab 4, step 5.6", "main.ipynb does not use src.stats yet → write your stats section in main.ipynb (from src.stats import ...).")
    except FileNotFoundError:
        _problem("Lab 2, Setup", "main.ipynb is missing → run the Setup cell of Lab 2 again (it creates main.ipynb), then add your Lab 4 stats section.")
    except Exception:
        _problem("Lab 2, Setup", "main.ipynb is empty or broken → rename it to main_old.ipynb, run the Setup cell of Lab 2 again, then add your Lab 4 stats section.")

if not STATS_OK:
    mean, count_by, group_stat = _ref_mean, _ref_count_by, _ref_group_stat
PROJECT_OK = not PROBLEMS

# ---------------------------------------------------------------- friendly error messages
def _friendly_error(shell, etype, evalue, tb, tb_offset=None):
    if etype is ProjectNotReady:
        print(evalue)
        return
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    msg = str(evalue)
    if "ellipsis" in msg:
        tip = "There is still a ... somewhere – replace it with your code."
    elif etype is NameError:
        tip = "Python does not know this name. Did you run the cell where it is created (▶)? Capital letters count."
    elif "NoneType" in msg or "returned non-string" in msg:
        tip = "Something gave None – does your method/function end with return ...?"
    elif etype is TypeError and "positional argument" in msg:
        tip = "Wrong number of values. Check that the first parameter in def is self, and compare the call with the def line."
    elif etype is ImportError:
        tip = "Your src/ file does not have this name yet (or it is spelled differently) → paste the classes into src/models.py (4.1), Ctrl+S, run again."
    elif etype is AttributeError:
        tip = ("This object has no such attribute or method. Check the spelling and that __init__ saves it as self.<name> = ...\n"
               "   Changed a class? Then create the objects again – old objects keep the old class.")
    else:
        tip = None
    if tip:
        print("\n💡", tip)

try:
    get_ipython().set_custom_exc((ProjectNotReady, NameError, TypeError, AttributeError, ImportError), _friendly_error)
except Exception:
    pass

# ---------------------------------------------------------------- report
print("=" * 70)
print(f"🧪 TEST DATA (Pokémon – NOT your data): {len(TEST)} rows in TEST")
print("Your project:", PROJECT)
if PROJECT_OK:
    print(f"  ✅ Lab 2: loader.py reads your data → {len(MY_ROWS)} rows in MY_ROWS")
    print("  ✅ Lab 4: stats.py: mean, median, count_by, group_stat work")
    print("  ✅ Lab 4: main.ipynb uses src.stats")
    _text = [c for c, t in MY_TYPES.items() if t in (str, bool)]
    _nums = [c for c, t in MY_TYPES.items() if t in (int, float)]
    print("  Your columns – text:", _text, "| numbers:", _nums)
    for n in NOTES:
        print("  ℹ️", n)
    print("🎉 Project ready → you can do all parts 2–5.")
else:
    print("❌ Your project is NOT ready for Lab 5. Fix these points first (in this order):")
    for i, p in enumerate(PROBLEMS, 1):
        print(f"  {i}. {p}")
    print("➡ Parts 2 and 3 work anyway (TEST DATA) – you can start them now.")
    print("  Parts 4–5 wait until the list is empty: fix it, then run this Setup cell again.")
    if not STATS_OK:
        print("  ℹ️ For parts 2–3 you get reference versions of mean, count_by, group_stat.")
print("=" * 70)

🧪 TEST DATA (Pokémon – NOT your data): 800 rows in TEST
Your project: c:\AGH sem7\pythonindatascience\PyDataProject
  ✅ Lab 2: loader.py reads your data → 27901 rows in MY_ROWS
  ✅ Lab 4: stats.py: mean, median, count_by, group_stat work
  ✅ Lab 4: main.ipynb uses src.stats
  Your columns – text: ['Gender', 'Sleep Duration', 'Depression'] | numbers: ['Age', 'CGPA', 'Financial Stress']
🎉 Project ready → you can do all parts 2–5.


## 2. Classes step by step (🧪 TEST DATA – Pokémon, not your data)
Until now one Pokémon was a **dictionary**:
```python
row = {"Name": "Pikachu", "Type 1": "Electric", "HP": 35, "Attack": 55, "Defense": 40, "Speed": 90, ...}
row["Attack"]     # 55
```
A dictionary only **stores** values. A **class** stores values **and** knows what to do with them.

| Word | Meaning | Example |
|---|---|---|
| **class** | a blueprint – like an empty form with fields Name, Type, HP… | `class Pokemon:` |
| **object** (instance) | one filled-in form made from the blueprint. One class → many objects | `pikachu`, `squirtle` |
| **attribute** | a value stored inside the object | `pikachu.attack` |
| **method** | a function that belongs to the class and works on the object | `pikachu.describe()` |

You have used objects from the first lab: in `"abc".upper()` the text `"abc"` is an object of the class `str` and `upper` is its method; in `rows.append(x)` – `append` is a method of the class `list`. Today you write **your own** classes.

Run the cell below – it shows what is in the TEST DATA.

In [2]:
# 🧪 TEST DATA (not your data): a list of dictionaries, one per Pokémon
print(len(TEST), "rows")
print(TEST[0])
print(TEST[24])
print(type(TEST[0]), type("abc"), type([1, 2]))   # every value is an object of some class

800 rows
{'Name': 'Bulbasaur', 'Type 1': 'Grass', 'Type 2': 'Poison', 'HP': 45, 'Attack': 49, 'Defense': 49, 'Speed': 45, 'Generation': 1, 'Legendary': False}
{'Name': 'Rattata', 'Type 1': 'Normal', 'Type 2': None, 'HP': 30, 'Attack': 56, 'Defense': 35, 'Speed': 72, 'Generation': 1, 'Legendary': False}
<class 'dict'> <class 'str'> <class 'list'>


### 2.1 Your first class: `__init__` and `self`
**Example (songs, not Pokémon):**
```python
class Song:
    def __init__(self, title, artist, seconds):
        self.title = title                       # attribute: the value is saved INSIDE the object
        self.artist = artist
        self.seconds = seconds
        self.minutes = round(seconds / 60, 1)    # computed attribute – calculated once, when the object is created

hit = Song("Bohemian Rhapsody", "Queen", 354)
print(hit.title, hit.minutes)                    # Bohemian Rhapsody 5.9
```
**What happens in `hit = Song("Bohemian Rhapsody", "Queen", 354)` – step by step**
1. Python creates a new, empty object of the class `Song`.
2. Python calls `__init__` **automatically** and gives it: `self` = this new object, `title` = `"Bohemian Rhapsody"`, `artist` = `"Queen"`, `seconds` = `354`. You never pass `self` yourself – Python does it.
3. `self.title = title` writes the value **into the object**. (Without `self.` it would be an ordinary variable that disappears when `__init__` ends.)
4. `self.minutes = round(354 / 60, 1)` → `5.9` is computed and stored too.
5. The ready object goes into the variable `hit`. Now `hit.title` reads the attribute.

**Remember:** `self` = "this particular object". Outside the class you write `hit.minutes`, inside the class `self.minutes` – it is the same thing.

**Typical mistakes**
| You wrote | Python says |
|---|---|
| `def __init__(title, artist, seconds):` (no `self`) | `TypeError: ... takes 3 positional arguments but 4 were given` |
| `title = title` instead of `self.title = title` | `AttributeError: 'Song' object has no attribute 'title'` |
| `_init_` or `__init` (wrong underscores) | `TypeError: Song() takes no arguments` – two underscores on **each** side! |

📌 **Under your nose:** [docs.python.org/3/tutorial/classes.html → 9.3.2 Class Objects](https://docs.python.org/3/tutorial/classes.html#class-objects) (scroll to the part about `__init__`) and [9.3.3 Instance Objects](https://docs.python.org/3/tutorial/classes.html#instance-objects).

**✏️ Task 2.1 a** – write the class `Pokemon`:
- `__init__(self, name, type1, hp, attack, defense, speed)` saves all **six** values as attributes with the same names (`self.name`, `self.type1`, …),
- plus one **computed** attribute `power` = hp + attack + defense + speed.

Expected: `Pokemon("Test", "Fire", 1, 2, 3, 4).power` → `10`.

In [3]:
class Pokemon:
    def __init__(self, name, type1, hp, attack, defense, speed):
        self.name = name        # ← done for you: save the name INSIDE the object
        # Step 1: do the same for the other five values – one line each:
        #         self.type1 = type1,  self.hp = hp,  ... attack, defense, speed
        self.type1 = type1
        self.hp = hp
        self.attack = attack
        self.defense = defense
        self.speed = speed
        # Step 2: one extra attribute: self.power = the sum of hp, attack, defense and speed
        self.power = self.hp + self.attack + self.defense + self.speed

# Step 3: run the cell. You should see:  Test 10
test_pokemon = Pokemon("Test", "Fire", 1, 2, 3, 4)
print(test_pokemon.name, test_pokemon.power)

Test 10


**✏️ Task 2.1 b** – make objects from the TEST rows:
1. Write a function `find(name)` that goes through `TEST` with a loop and **returns the row** whose `"Name"` is `name` (and `None` if there is no such Pokémon).
2. Create the object `pikachu` from the row of `"Pikachu"`. Careful: the order of values must match the order in `__init__` (name, type1, hp, attack, defense, speed) and the keys in the row are `"Name"`, `"Type 1"`, `"HP"`, `"Attack"`, `"Defense"`, `"Speed"`.
3. Make the list `team` with Pokémon **objects** made from the first 6 rows of `TEST` (loop + `append`).

Expected: `pikachu.power` → `220`; `[p.power for p in team]` → `[188, 245, 325, 383, 199, 260]`.

<details><summary>💡 Hint</summary>

`find`: the same pattern as searching in a list in Lab 3 – `for row in TEST:` → `if row["Name"] == name:` → `return row`; after the loop `return None`.
Object from a row: `Pokemon(r["Name"], r["Type 1"], r["HP"], ...)` where `r = find("Pikachu")`.
</details>

In [4]:
def find(name):
    """Return the row of TEST with this Name (None if there is none)."""
    # Step 1: go through all rows:                 for row in TEST:
    # Step 2: is it the row we look for?             if row["Name"] == name:
    # Step 3: yes → give it back:                    return row
    # Step 4: after the loop (nothing found):        return None
    for row in TEST:
        if row["Name"] == name:
            return row
    return None
r = find("Pikachu")          # r = the row (a dictionary) of Pikachu
print(r)

# Step 5: make the OBJECT. Pokemon(...) needs 6 values in the order of __init__:
#         name, type1, hp, attack, defense, speed  →  take them from r: r["Name"], r["Type 1"], r["HP"], ...
pikachu = Pokemon(r["Name"], r["Type 1"], r["HP"], r["Attack"], r["Defense"], r["Speed"])
print(pikachu.name, pikachu.type1, "power:", pikachu.power)      # expected: Pikachu Electric power: 220

# Step 6: a team of 6 objects. The loop gives you the first 6 rows one by one (row).
#         For each row: make a Pokemon exactly like in step 5, but from row instead of r,
#         and add it to the list:   team.append( Pokemon(row["Name"], ...) )
team = []
for row in TEST[:6]:
    team.append(Pokemon(row["Name"], row["Type 1"], row["HP"], row["Attack"], row["Defense"], row["Speed"]))
print([p.power for p in team])          # expected: [188, 245, 325, 383, 199, 260]

{'Name': 'Pikachu', 'Type 1': 'Electric', 'Type 2': None, 'HP': 35, 'Attack': 55, 'Defense': 40, 'Speed': 90, 'Generation': 1, 'Legendary': False}
Pikachu Electric power: 220
[188, 245, 325, 383, 199, 260]


**🤔 Predict before you run:** what will `print(team[0])` show? Write your guess in the comment, then run.

In [5]:
print(team[0])      # my guess: 0x
# Python shows only "an object of class Pokemon at this memory address" – in 2.3 you teach it to print nicely.

In [6]:
# @title ✔️ Check 2.1
check("2.1 a class Pokemon saves the six values",
      lambda: (lambda p: (p.name, p.type1, p.hp, p.attack, p.defense, p.speed) == ("X", "Fire", 1, 2, 3, 4))(Pokemon("X", "Fire", 1, 2, 3, 4)),
      "in __init__: self.name = name, self.type1 = type1, ... (six lines)")
check("2.1 a computed attribute power", lambda: Pokemon("X", "Fire", 1, 2, 3, 4).power == 10,
      "self.power = hp + attack + defense + speed")
check("2.1 b find('Pikachu') returns the row", lambda: find("Pikachu")["HP"] == 35 and find("Nobody") is None,
      "loop over TEST, return row when row['Name'] == name; after the loop return None")
check("2.1 b pikachu.power == 220", lambda: isinstance(pikachu, Pokemon) and pikachu.power == 220,
      "pikachu = Pokemon(r['Name'], r['Type 1'], r['HP'], r['Attack'], r['Defense'], r['Speed'])")
check("2.1 b team = 6 Pokemon objects", lambda: [p.power for p in team] == [188, 245, 325, 383, 199, 260] and all(isinstance(p, Pokemon) for p in team),
      "for row in TEST[:6]: team.append(Pokemon(...)) – values from row")
summary("2.1")

✅ 2.1 a class Pokemon saves the six values
✅ 2.1 a computed attribute power
✅ 2.1 b find('Pikachu') returns the row
✅ 2.1 b pikachu.power == 220
✅ 2.1 b team = 6 Pokemon objects

5/5 OK  🎉


### 2.2 Methods – functions inside the class
**Example (songs):**
```python
class Song:
    def __init__(self, title, artist, seconds):
        ...                                        # as in 2.1
    def label(self):
        return f"{self.artist} – {self.title} ({self.minutes} min)"
    def is_long(self, limit=300):                  # limit has a DEFAULT value
        return self.seconds > limit
    def longer_than(self, other):                  # other = ANOTHER Song object
        return self.seconds > other.seconds

hit = Song("Bohemian Rhapsody", "Queen", 354)
short = Song("Imagine", "John Lennon", 183)
hit.label()              # 'Queen – Bohemian Rhapsody (5.9 min)'
hit.is_long()            # True   (354 > 300)
hit.is_long(400)         # False  (354 > 400 is not true)
hit.longer_than(short)   # True   (354 > 183)
```
**What happens in `hit.is_long(400)` – step by step**
1. Python sees: object `hit`, method `is_long`. It looks for `is_long` in the class of `hit` → `Song`.
2. It calls it as `Song.is_long(hit, 400)` → inside: `self` = `hit`, `limit` = `400`.
3. `self.seconds` → `354`; `354 > 400` → `False` → `return False`.

A method = a function written **inside** the class (indented), whose **first parameter is `self`**. Through `self` it can read every attribute of the object. A method can also get **another object** as a parameter (`other`) and compare.

⚠️ **After you change a class, objects made earlier stay "old"** (they keep the old version without the new methods). Create them again – that's why in this task you write a function `make(row)` that creates an object from a row in one line.

📌 **Under your nose:** [classes.html → 9.3.4 Method Objects](https://docs.python.org/3/tutorial/classes.html#method-objects)

**✏️ Task 2.2 a** – the cell below already has the frame of the class. Follow the steps in the cell: **Step 1** copy your `__init__` from 2.1 a, **Steps 2–4** fill the three new methods:
| Method | Returns | Example |
|---|---|---|
| `describe(self)` | text `"<name> (<type1>), power <power>"` | `"Pikachu (Electric), power 220"` |
| `is_fast(self, limit=100)` | `True` when speed is **at least** `limit` | Pikachu (speed 90): `is_fast()` → `False`, `is_fast(90)` → `True` |
| `beats(self, other)` | `True` when **this** attack is bigger than the **other's defense** | Snorlax (attack 110) beats Gyarados (defense 79) → `True` |

In [7]:
class Pokemon:
    # Step 1: copy here your whole  def __init__(...)  from 2.1 a (with all its lines)
    def __init__(self, name, type1, hp, attack, defense, speed):
            self.name = name        # ← done for you: save the name INSIDE the object
            # Step 1: do the same for the other five values – one line each:
            #         self.type1 = type1,  self.hp = hp,  ... attack, defense, speed
            self.type1 = type1
            self.hp = hp
            self.attack = attack
            self.defense = defense
            self.speed = speed
            # Step 2: one extra attribute: self.power = the sum of hp, attack, defense and speed
            self.power = self.hp + self.attack + self.defense + self.speed

    # Step 2: return the text  "Pikachu (Electric), power 220"  – use self.name, self.type1, self.power
    #         hint: return f"{self.name} (...), power ..."
    def describe(self):
        return f"{self.name} ({self.type1}), power {self.power}"

    # Step 3: True when the speed is AT LEAST limit (>=), otherwise False
    #         hint: return self.speed ... limit
    def is_fast(self, limit=100):
        return self.speed >= limit

    # Step 4: True when MY attack is bigger than the OTHER Pokémon's defense
    #         hint: return self.attack ... other.defense
    def beats(self, other):
        return self.attack > other.defense

**✏️ Task 2.2 b** – use the methods (write code, do not count by hand):
1. `make(row)` – a function that returns a `Pokemon` object made from one row of TEST (one line with `return`).
2. `all_pokemon` – a list of objects made from **all** 800 rows.
3. `n_fast` – how many of them are fast with the default limit; `n_very_fast` – how many with limit 120. Use the method `is_fast`, not `p.speed >= ...`.
4. Does Pikachu beat Squirtle? Does Snorlax beat Gyarados? (use `find`, `make` and `beats`)

Expected: `n_fast` → `137`, `n_very_fast` → `34`, Pikachu vs Squirtle → `False`, Snorlax vs Gyarados → `True`.

In [8]:
# Step 1: make(row) = create a Pokemon from one row – the same as step 5 in 2.1 b, but with row
def make(row):
    """Create a Pokemon object from one row of TEST."""
    return Pokemon(row["Name"], row["Type 1"], row["HP"], row["Attack"], row["Defense"], row["Speed"])

pikachu = make(find("Pikachu"))          # made again → it has the new methods
print(pikachu.describe(), "| fast:", pikachu.is_fast(), "| fast for limit 90:", pikachu.is_fast(90))

# Step 2: all 800 Pokémon as objects: for every row in TEST → all_pokemon.append(make(row))
all_pokemon = []
for row in TEST:
    all_pokemon.append(make(row))

# Step 3: count. For every Pokémon p:
#         if p.is_fast():      n_fast += 1
#         if p.is_fast(120):   n_very_fast += 1
n_fast = 0
n_very_fast = 0
for p in all_pokemon:
    if p.is_fast():
        n_fast += 1
        if p.is_fast(120):
            n_very_fast += 1
print("fast:", n_fast, "| very fast:", n_very_fast)      # expected: 137 | 34

# Step 4: who beats whom?  make(find("Pikachu")).beats( make(find("Squirtle")) )
pikachu_beats_squirtle = make(find("Pikachu")).beats(make(find("Squirtle")))
snorlax_beats_gyarados = make(find("Snorlax")).beats(make(find("Gyarados")))
print(pikachu_beats_squirtle, snorlax_beats_gyarados)      # expected: False True

Pikachu (Electric), power 220 | fast: False | fast for limit 90: True
fast: 137 | very fast: 34
False True


In [9]:
# @title ✔️ Check 2.2
_p = lambda: Pokemon("Pikachu", "Electric", 35, 55, 40, 90)
check("2.2 a describe()", lambda: _p().describe() == "Pikachu (Electric), power 220",
      'return f"{self.name} ({self.type1}), power {self.power}"  – look at the comma and the brackets')
check("2.2 a is_fast(limit=100) uses >= and the default", lambda: _p().is_fast() is False and _p().is_fast(90) is True,
      "def is_fast(self, limit=100):  return self.speed >= limit")
check("2.2 a beats(other) = my attack > other's defense",
      lambda: Pokemon("A", "x", 1, 80, 1, 1).beats(Pokemon("B", "x", 1, 1, 79, 1)) is True and Pokemon("A", "x", 1, 79, 1, 1).beats(Pokemon("B", "x", 1, 1, 79, 1)) is False,
      "return self.attack > other.defense")
check("2.2 b make(row) and all_pokemon", lambda: make(TEST[0]).power == 188 and len(all_pokemon) == 800 and hasattr(all_pokemon[0], "is_fast"),
      "make: return Pokemon(row['Name'], ...);  all_pokemon: loop over TEST and append(make(row))")
check("2.2 b n_fast == 137 and n_very_fast == 34", lambda: (n_fast, n_very_fast) == (137, 34),
      "in the loop: if p.is_fast(): n_fast += 1  and  if p.is_fast(120): n_very_fast += 1")
check("2.2 b who beats whom", lambda: (pikachu_beats_squirtle, snorlax_beats_gyarados) == (False, True),
      'make(find("Pikachu")).beats(make(find("Squirtle")))')
summary("2.2")

✅ 2.2 a describe()
✅ 2.2 a is_fast(limit=100) uses >= and the default
✅ 2.2 a beats(other) = my attack > other's defense
✅ 2.2 b make(row) and all_pokemon
✅ 2.2 b n_fast == 137 and n_very_fast == 34
✅ 2.2 b who beats whom

6/6 OK  🎉


### 2.3 "Function overloading" in Python
In some languages (Java, C++) you can write **two functions with the same name but different parameters**, and the language picks the right one – this is called *overloading*. **Python does not do that:** a second `def` with the same name simply **replaces** the first one. Predict what happens, then run:

In [10]:
def area(r):
    return 3.14 * r * r              # area of a circle

def area(width, height):             # the same name → this one REPLACES the first area!
    return width * height            # area of a rectangle

print(area(3, 4))                    # my guess: 12
try:                                 # try/except catches the error so the notebook keeps running (Lab 6)
    print(area(2))                   # my guess: 4
except TypeError as e:
    print("TypeError:", e)

12
TypeError: area() missing 1 required positional argument: 'height'


**So how do we "overload" in Python? Two ways:**

**1. Default values of parameters** – one function, several ways of calling it. You already did it: `is_fast()` and `is_fast(120)`. The area example the Python way:
```python
def area(a, b=None):
    if b is None:              # called with one value → circle
        return 3.14 * a * a
    return a * b               # called with two values → rectangle
```

**2. Special methods (operator overloading)** – methods with names like `__str__`, `__eq__`, `__lt__`. You never call them yourself: **Python calls them automatically** when you use `print`, `==`, `<`, `sorted` … on your object.

| You write | Python calls |
|---|---|
| `print(p)`, `str(p)` | `p.__str__()` – must **return** a text |
| `p1 == p2` | `p1.__eq__(p2)` |
| `p1 < p2`, and also `sorted(list)`, `max(list)`, `min(list)` | `p1.__lt__(p2)` ("lt" = *less than*) |
| `len(x)` | `x.__len__()` |
| `a + b` | `a.__add__(b)` |

**Example (songs):**
```python
class Song:
    ...                                          # __init__, label, is_long as before
    def __str__(self):
        return self.label()                      # print(hit) → Queen – Bohemian Rhapsody (5.9 min)
    def __eq__(self, other):
        return self.title == other.title and self.artist == other.artist
    def __lt__(self, other):
        return self.seconds < other.seconds      # "smaller" song = shorter song
```
**What happens in `sorted(songs)` – step by step:** `sorted` must compare pairs of songs → for each pair it asks "is a < b?" → Python calls `a.__lt__(b)` → compares `seconds` → result: list from the **shortest** song. `max(songs)` uses the same `<` → the **longest** song.

Without `__eq__`, `==` only checks whether it is **the same object in memory** – two separate objects with identical data are "not equal".

📌 **Under your nose:** default values – [tutorial/controlflow.html → 4.9.1 Default Argument Values](https://docs.python.org/3/tutorial/controlflow.html#default-argument-values); special methods – [reference/datamodel.html → `object.__str__`](https://docs.python.org/3/reference/datamodel.html#object.__str__) and a bit lower [`object.__lt__` / `__eq__`](https://docs.python.org/3/reference/datamodel.html#object.__lt__) ("rich comparison" methods).

**✏️ Task 2.3 a** – the cell below has the frame of the class. **Step 1:** copy `__init__`, `is_fast` and `beats` from 2.2 a. **Steps 2–5:** fill the methods from this table:
| What | Rule | Example |
|---|---|---|
| `describe(self, short=False)` | change it: `describe()` gives the long text as before, `describe(short=True)` gives only `"<name> (<type1>)"` | `"Pikachu (Electric)"` |
| `__str__(self)` | returns the long description (use `describe`, do not write the text twice) | `print(pikachu)` → `Pikachu (Electric), power 220` |
| `__eq__(self, other)` | two Pokémon are equal when they have **the same name** | |
| `__lt__(self, other)` | "smaller" = **lower power** → `sorted()` goes from the weakest, `max()` gives the strongest | |

In [11]:
class Pokemon:
    # Step 1: copy here from 2.2 a:  def __init__,  def is_fast,  def beats   (NOT describe – it changes below)
    def __init__(self, name, type1, hp, attack, defense, speed):
        self.name = name
        self.type1 = type1
        self.hp = hp
        self.attack = attack
        self.defense = defense
        self.speed = speed
        self.power = self.hp + self.attack + self.defense + self.speed

    def is_fast(self, limit=100):
        return self.speed >= limit

    def beats(self, other):
        return self.attack > other.defense

    # Step 2: one method, two ways of calling it:
    #         describe()           → "Pikachu (Electric), power 220"
    #         describe(short=True) → "Pikachu (Electric)"
    def describe(self, short=False):
        if short:
            return f"{self.name} ({self.type1})"
        return f"{self.name} ({self.type1}), power {self.power}"

    # Step 3: print(p) calls this – give back the long description:  return self.describe()
    def __str__(self):
        return self.describe()

    # Step 4: p1 == p2 calls this – equal when the names are the same:  self.name == other.name
    def __eq__(self, other):
        return self.name == other.name

    # Step 5: p1 < p2, sorted(), max() call this – "smaller" = lower power:  self.power < other.power
    def __lt__(self, other):
        return self.power < other.power

**✏️ Task 2.3 b** – first **predict** in the comments, then run:
1. Create `team` again from the first 6 rows (with `make`) – what does `print(team[0])` show now?
2. `sorted(team)` – which name is first and which is last?
3. `max(all_pokemon)` – the strongest of all 800 (create `all_pokemon` again!).
4. `make(find("Pikachu")) == make(find("Pikachu"))` – two different objects: `True` or `False`?

Expected: sorted names `['Bulbasaur', 'Charmander', 'Ivysaur', 'Charmeleon', 'Venusaur', 'VenusaurMega Venusaur']`; strongest `GroudonPrimal Groudon (Ground), power 530`; `==` → `True`.

In [12]:
# The class changed → make the objects AGAIN (old objects keep the old class).
# Step 1: team = 6 objects from the first 6 rows – use make(row) in a loop (like in 2.1 b) or in one line:
#         [make(row) for row in TEST[:6]]
team = [make(row) for row in TEST[:6]]
print(team[0])                                   # my guess: this time we get a description instead of address
print(team[0].describe(short=True))

# Step 2: sort the team:  sorted(team)
weakest_to_strongest = sorted(team)
print([p.name for p in weakest_to_strongest])    # my guess first / last: bulbasaur / venusaur

# Step 3: all 800 again (like step 2 in 2.2 b), then the strongest:  max(all_pokemon)
all_pokemon = [make(row) for row in TEST]
strongest = max(all_pokemon)
print("strongest:", strongest)

# Step 4: two SEPARATE Pikachu objects – are they equal?  make(find("Pikachu")) == make(find("Pikachu"))
same = make(find("Pikachu")) == make(find("Pikachu"))
print("== :", same)                              # my guess: True

Bulbasaur (Grass), power 188
Bulbasaur (Grass)
['Bulbasaur', 'Charmander', 'Ivysaur', 'Charmeleon', 'Venusaur', 'VenusaurMega Venusaur']
strongest: GroudonPrimal Groudon (Ground), power 530
== : True


In [13]:
# @title ✔️ Check 2.3
_p = lambda: Pokemon("Pikachu", "Electric", 35, 55, 40, 90)
check("2.3 a describe() long and describe(short=True)",
      lambda: _p().describe() == "Pikachu (Electric), power 220" and _p().describe(short=True) == "Pikachu (Electric)",
      "def describe(self, short=False):  if short: return f\"{self.name} ({self.type1})\"  – otherwise the long text")
check("2.3 a __str__ → print shows the description", lambda: str(_p()) == "Pikachu (Electric), power 220",
      "def __str__(self): return self.describe()")
check("2.3 a __eq__ compares names", lambda: (_p() == _p()) is True and (_p() == Pokemon("Raichu", "Electric", 1, 1, 1, 1)) is False,
      "def __eq__(self, other): return self.name == other.name")
check("2.3 a __lt__ compares power", lambda: (Pokemon("A", "x", 1, 1, 1, 1) < Pokemon("B", "x", 2, 2, 2, 2)) is True,
      "def __lt__(self, other): return self.power < other.power")
check("2.3 b sorted(team)", lambda: [p.name for p in weakest_to_strongest] == ["Bulbasaur", "Charmander", "Ivysaur", "Charmeleon", "Venusaur", "VenusaurMega Venusaur"],
      "make team again with the NEW class, then weakest_to_strongest = sorted(team)")
check("2.3 b max(all_pokemon)", lambda: str(strongest) == "GroudonPrimal Groudon (Ground), power 530",
      "make all_pokemon again with the NEW class, then strongest = max(all_pokemon)")
check("2.3 b == of two separate objects", lambda: same is True, 'same = make(find("Pikachu")) == make(find("Pikachu"))')
summary("2.3")

✅ 2.3 a describe() long and describe(short=True)
✅ 2.3 a __str__ → print shows the description
✅ 2.3 a __eq__ compares names
✅ 2.3 a __lt__ compares power
✅ 2.3 b sorted(team)
✅ 2.3 b max(all_pokemon)
✅ 2.3 b == of two separate objects

7/7 OK  🎉


### 2.4 Class attribute vs. instance attribute
- **Instance attribute** – `self.x = ...` in `__init__`: **every object has its own** value (each Pokémon has its own `hp`).
- **Class attribute** – written directly in the class, **not** in a method: **one shared value** for the whole class. All objects see it.

**Example:**
```python
class Song:
    played = 0                    # class attribute – ONE counter for all songs
    def __init__(self, title):
        self.title = title        # instance attribute – each song has its own title
    def play(self):
        Song.played += 1          # change the SHARED value: ClassName.attribute
```
**🤔 Predict** the results in the comments of the next cell, then run it. The second class `Trap` shows the most common mistake.

In [14]:
class Song:
    played = 0
    def __init__(self, title):
        self.title = title
    def play(self):
        Song.played += 1

a, b = Song("Hey Jude"), Song("Imagine")
a.play(); a.play(); b.play()
print(Song.played)     # my guess: 3
print(a.played)        # my guess: 3

class Trap:
    played = 0
    def play(self):
        self.played += 1   # self. instead of Trap. → creates a NEW own attribute of this one object!

t1, t2 = Trap(), Trap()
t1.play(); t1.play(); t2.play()
print(Trap.played, t1.played, t2.played)   # my guess: 0, 2, 1

3
3
0 2 1


**✏️ Task 2.4** – a ticket machine ("take a number") at an office: write the class `Ticket`:
- class attribute `next_number = 1`,
- `__init__(self, person)`: saves `person`, gives the ticket `self.number` = the current `Ticket.next_number`, **then** increases `Ticket.next_number` by 1,
- `__str__` → `"Ticket 2: Ben"`.

Then create three tickets `t1`, `t2`, `t3` for `"Anna"`, `"Ben"`, `"Ola"` (keep them in the same cell as the class – when you run the cell again, the counter starts again from 1).

Expected: numbers `1, 2, 3`; `Ticket.next_number` → `4`; `print(t2)` → `Ticket 2: Ben`.

In [15]:
class Ticket:
    # Step 1: class attribute (written in the class, NOT in __init__) – the first number is 1
    next_number = 1

    def __init__(self, person):
        # Step 2: save the person:                          self.person = person
        # Step 3: this ticket gets the CURRENT number:        self.number = Ticket.next_number
        # Step 4: the next ticket gets the next number:       Ticket.next_number += 1   (Ticket. – not self.!)
        self.person = person
        self.number = Ticket.next_number
        Ticket.next_number += 1

    # Step 5: print(t2) → "Ticket 2: Ben"   hint: f"Ticket {self.number}: ..."
    def __str__(self):
        return f"Ticket {self.number}: {self.person}"

# Step 6: three tickets:  Ticket("Anna"), Ticket("Ben"), Ticket("Ola")
t1 = Ticket("Anna")
t2 = Ticket("Ben")
t3 = Ticket("Ola")
print(t1, "|", t2, "|", t3, "| next:", Ticket.next_number)      # expected: Ticket 1: Anna | Ticket 2: Ben | Ticket 3: Ola | next: 4

Ticket 1: Anna | Ticket 2: Ben | Ticket 3: Ola | next: 4


In [16]:
# @title ✔️ Check 2.4
check("2.4 t1, t2, t3 have numbers 1, 2, 3", lambda: (t1.number, t2.number, t3.number) == (1, 2, 3),
      "self.number = Ticket.next_number, then Ticket.next_number += 1 (run the whole cell again)")
check("2.4 the counter is a CLASS attribute", lambda: "next_number" in vars(Ticket) and "next_number" not in vars(t1),
      "write next_number = 1 directly in the class and change it with Ticket.next_number (not self.)")
def _new_ticket():
    n = Ticket.next_number
    x = Ticket("X")
    ok = x.number == n and Ticket.next_number == n + 1
    Ticket.next_number = n
    return ok
check("2.4 every new ticket gets the next number", _new_ticket, "Ticket.next_number += 1 inside __init__")
check("2.4 __str__", lambda: str(t2) == "Ticket 2: Ben", 'return f"Ticket {self.number}: {self.person}"')
summary("2.4")

✅ 2.4 t1, t2, t3 have numbers 1, 2, 3
✅ 2.4 the counter is a CLASS attribute
✅ 2.4 every new ticket gets the next number
✅ 2.4 __str__

4/4 OK  🎉


### 2.5 Inheritance – "a special kind of …"
A legendary Pokémon **is** a Pokémon, only with something extra and a different description. Instead of copying the whole class you write:
```python
class Legendary(Pokemon):     # Legendary = a Pokemon + changes
```
- The **child class** (`Legendary`) gets **all** attributes and methods of the **parent class** (`Pokemon`). You write only what is **new** or **different**.
- **`super()`** = "the parent class". `super().__init__(...)` = let the parent do its normal job (save name, type, …, compute power), then add the extra.
- **Overriding** = a method with the same name in the child **replaces** the parent's one (for child objects). Inside it you can still use the parent's version with `super().method(...)`.

**Example (songs):**
```python
class Podcast(Song):                                   # a Podcast IS a Song + a host
    def __init__(self, title, artist, seconds, host):
        super().__init__(title, artist, seconds)        # the parent saves title, artist, seconds, minutes
        self.host = host                                # the only new thing
    def label(self):                                    # override: a different label
        return super().label() + f" – hosted by {self.host}"

ep = Podcast("Data talk", "Radio X", 1800, "Kasia")
ep.minutes      # 30.0  – computed by Song.__init__
ep.is_long()    # True  – method of Song, not written in Podcast at all
ep.label()      # 'Radio X – Data talk (30.0 min) – hosted by Kasia'
```
**How Python finds a method – step by step (`ep.is_long()`):**
1. Look in the class of the object → `Podcast`. Is there `is_long`? **No.**
2. Look in the parent → `Song`. Is there `is_long`? **Yes** → use it.

For `ep.label()` Python finds `label` already in `Podcast` (step 1) → the child's version wins. Inside it, `super().label()` runs the parent's version and the child adds the host.
`isinstance(ep, Song)` → `True` (a podcast is also a song); `isinstance(hit, Podcast)` → `False`.

📌 **Under your nose:** [classes.html → 9.5 Inheritance](https://docs.python.org/3/tutorial/classes.html#inheritance), [functions.html → super()](https://docs.python.org/3/library/functions.html#super), [functions.html → isinstance()](https://docs.python.org/3/library/functions.html#isinstance)

**✏️ Task 2.5** – write the child class `Legendary(Pokemon)`. **Do not copy the class Pokemon** – write only what is new:
- `__init__(self, name, type1, hp, attack, defense, speed, generation)` → the parent saves the first six (use `super()`), you save `generation`,
- override `describe(self, short=False)` → the parent's text (long or short) + `" ★ legendary, generation <generation>"`.

Then create `mewtwo` from the TEST row of `"Mewtwo"` (the generation is in the column `"Generation"`). The steps are in the cell.

**🤔 Predict:** you do **not** write `__str__` in `Legendary`. What will `print(mewtwo)` show – with the star or without? (`__str__` of the parent calls `self.describe()` … and `self` is Mewtwo.)

Expected: `mewtwo.describe()` → `Mewtwo (Psychic), power 436 ★ legendary, generation 1`; `mewtwo.is_fast()` → `True` (inherited).

In [17]:
class Legendary(Pokemon):          # Legendary = a Pokemon + changes. Do NOT copy the class Pokemon!
    def __init__(self, name, type1, hp, attack, defense, speed, generation):
        # Step 1: let the PARENT save the six values:  super().__init__(name, type1, hp, attack, defense, speed)
        super().__init__(name, type1, hp, attack, defense, speed)
        # Step 2: save the only new thing:  self.generation = generation
        self.generation = generation

    # Step 3: take the PARENT's text and add the star:
    #         super().describe(short)  +  f" ★ legendary, generation {self.generation}"
    def describe(self, short=False):
        return super().describe(short) + f" ★ legendary, generation {self.generation}"

# Step 4: make mewtwo like pikachu in 2.1 b, but with Legendary(...) and r["Generation"] as the 7th value
r = find("Mewtwo")
mewtwo = Legendary(r["Name"], r["Type 1"], r["HP"], r["Attack"], r["Defense"], r["Speed"], r["Generation"])
print(mewtwo.describe())             # expected: Mewtwo (Psychic), power 436 ★ legendary, generation 1
print(mewtwo.describe(short=True))
print(mewtwo)                        # my guess – star or no star?
print(mewtwo.is_fast(), isinstance(mewtwo, Pokemon), isinstance(make(find("Pikachu")), Legendary))

Mewtwo (Psychic), power 436 ★ legendary, generation 1
Mewtwo (Psychic) ★ legendary, generation 1
Mewtwo (Psychic), power 436 ★ legendary, generation 1
True True False


In [18]:
# @title ✔️ Check 2.5
check("2.5 Legendary is a child of Pokemon", lambda: issubclass(Legendary, Pokemon) and Legendary is not Pokemon,
      "class Legendary(Pokemon):")
check("2.5 no copy: Legendary inherits is_fast, beats, __lt__", lambda: not any(m in vars(Legendary) for m in ["is_fast", "beats", "__lt__", "__eq__"]),
      "write in Legendary ONLY __init__ and describe – the rest comes from Pokemon")
check("2.5 super().__init__ did the parent's job", lambda: mewtwo.power == 436 and mewtwo.name == "Mewtwo" and mewtwo.generation == 1,
      "super().__init__(name, type1, hp, attack, defense, speed)  then  self.generation = generation")
check("2.5 describe() long and short",
      lambda: mewtwo.describe() == "Mewtwo (Psychic), power 436 ★ legendary, generation 1" and mewtwo.describe(short=True) == "Mewtwo (Psychic) ★ legendary, generation 1",
      'return super().describe(short) + f" ★ legendary, generation {self.generation}"')
check("2.5 print(mewtwo) uses the CHILD's describe", lambda: str(mewtwo).endswith("generation 1"),
      "do not write __str__ in Legendary – the parent's __str__ calls self.describe()")
check("2.5 inherited methods work", lambda: mewtwo.is_fast() is True and isinstance(mewtwo, Pokemon), "")
summary("2.5")

✅ 2.5 Legendary is a child of Pokemon
✅ 2.5 no copy: Legendary inherits is_fast, beats, __lt__
✅ 2.5 super().__init__ did the parent's job
✅ 2.5 describe() long and short
✅ 2.5 print(mewtwo) uses the CHILD's describe
✅ 2.5 inherited methods work

6/6 OK  🎉


### 2.6 Polymorphism – one call, many behaviours
*Poly-morph* = "many shapes". The **same call** (`x.describe()`) works on objects of **different classes**, and **each object answers in its own way**. The loop does not need an `if` for every type – the object itself knows its class.

**Example (songs):**
```python
playlist = [Song("Hey Jude", "The Beatles", 431),
            Podcast("Data talk", "Radio X", 1800, "Kasia"),
            Song("Imagine", "John Lennon", 183)]
for item in playlist:
    print(item.label())        # Song.label or Podcast.label – Python picks it by the class of item
```
Without polymorphism you would need `if item is a podcast: ... else: ...` **everywhere** you use the list. With polymorphism the `if` is in **one** place: where the objects are created.

`type(x).__name__` gives the name of the class of an object as text, e.g. `type(ep).__name__` → `'Podcast'`.

**✏️ Task 2.6** – a mixed zoo:
1. Write a second child class `Mega(Pokemon)` (Mega evolutions). It needs **no** `__init__` (it inherits the parent's). Override:
   - `describe(self, short=False)` → `"MEGA "` + the parent's text,
   - `is_fast(self, limit=100)` → a Mega counts as fast with **20 points less**: `speed + 20 >= limit`.
2. Write `make_any(row)` – the **only** place with `if`: returns `Legendary(...)` when `row["Legendary"]` is `True`, otherwise `Mega(...)` when `"Mega "` is in the name, otherwise `Pokemon(...)`.
3. `zoo` = objects from all 800 rows made with `make_any`. Then **one loop without any `if` on the class** that:
   - counts objects per class name in the dictionary `per_class` (use `type(p).__name__`),
   - counts fast ones in `n_fast_zoo` with `p.is_fast()` – every object uses **its own** `is_fast`.

Expected: `per_class` → `{'Pokemon': 693, 'Mega': 42, 'Legendary': 65}` (order may differ); `n_fast_zoo` → `141` (in 2.2 it was 137 – 4 Megas are now fast thanks to their own `is_fast`).

<details><summary>💡 Hint</summary>

Counting in a dictionary – the same as `count_by` in Lab 4: `per_class[k] = per_class.get(k, 0) + 1` where `k = type(p).__name__`.
`make_any`: `if row["Legendary"]: return Legendary(..., row["Generation"])` → `if "Mega " in row["Name"]: return Mega(...)` → `return Pokemon(...)`.
Skrót: `values` ma już sześć wartości – `Pokemon(*values)` = `Pokemon(values[0], values[1], ..., values[5])` (gwiazdka je rozpakowuje).
Shortcut: `values` already holds the six values – `Pokemon(*values)` = `Pokemon(values[0], values[1], ..., values[5])` (the star unpacks them).
</details>

In [19]:
class Mega(Pokemon):               # no __init__ needed – it comes from Pokemon
    # Step 1: "MEGA " + the parent's text:   "MEGA " + super().describe(short)
    def describe(self, short=False):
        return "MEGA " + super().describe(short)

    # Step 2: a Mega is fast with 20 points less:   self.speed + 20 >= limit
    def is_fast(self, limit=100):
        return self.speed + 20 >= limit


def make_any(row):
    """The ONLY place with if: choose the class by the row."""
    values = (row["Name"], row["Type 1"], row["HP"], row["Attack"], row["Defense"], row["Speed"])
    # values = the six values. Pokemon(*values) = Pokemon(values[0], values[1], ..., values[5]) – the star unpacks them
    # Step 3: if row["Legendary"]:          return Legendary(*values, row["Generation"])
    # Step 4: if "Mega " in row["Name"]:      return Mega(*values)
    # Step 5: otherwise:                      return Pokemon(*values)
    if (row["Legendary"]):
        return Legendary(*values, row["Generation"])
    if ("Mega " in row["Name"]):
        return Mega(*values)
    return Pokemon(*values)


# Step 6: all 800 rows → objects with make_any (like all_pokemon in 2.3 b, but make_any instead of make)
zoo = [make_any(row) for row in TEST]
# Step 7: ONE loop, no "if" on the class. For every p:
#         k = type(p).__name__                      ← the class name as text: "Pokemon", "Mega" or "Legendary"
#         per_class[k] = per_class.get(k, 0) + 1    ← count it (like count_by in Lab 4)
#         if p.is_fast(): n_fast_zoo += 1           ← every object uses ITS OWN is_fast
per_class = {}
n_fast_zoo = 0
for p in zoo:
    k = type(p).__name__
    per_class[k] = per_class.get(k, 0) + 1
    if p.is_fast():
        n_fast_zoo += 1
print(per_class, "| fast:", n_fast_zoo)     # expected: {'Pokemon': 693, 'Mega': 42, 'Legendary': 65} | 141

for p in [zoo[0], zoo[3], zoo[156]]:
    print(p.describe())          # the same call – three different answers 

{'Pokemon': 693, 'Mega': 42, 'Legendary': 65} | fast: 141
Bulbasaur (Grass), power 188
MEGA VenusaurMega Venusaur (Grass), power 383
Articuno (Ice), power 360 ★ legendary, generation 1


In [20]:
# @title ✔️ Check 2.6
check("2.6 Mega is a child of Pokemon without its own __init__", lambda: issubclass(Mega, Pokemon) and "__init__" not in vars(Mega),
      "class Mega(Pokemon):  – write only describe and is_fast")
check("2.6 Mega.describe", lambda: Mega("M", "Fire", 1, 1, 1, 1).describe() == "MEGA M (Fire), power 4",
      'return "MEGA " + super().describe(short)')
check("2.6 Mega.is_fast with 20 points less", lambda: Mega("M", "x", 1, 1, 1, 80).is_fast() is True and Mega("M", "x", 1, 1, 1, 79).is_fast() is False,
      "return self.speed + 20 >= limit")
check("2.6 make_any chooses the right class", lambda: [type(make_any(find(n))).__name__ for n in ["Mewtwo", "VenusaurMega Venusaur", "Pikachu"]] == ["Legendary", "Mega", "Pokemon"],
      "first if row['Legendary'] → Legendary, then 'Mega ' in name → Mega, else Pokemon")
check("2.6 per_class", lambda: per_class == {"Pokemon": 693, "Mega": 42, "Legendary": 65},
      "k = type(p).__name__;  per_class[k] = per_class.get(k, 0) + 1")
check("2.6 n_fast_zoo == 141 (polymorphism!)", lambda: n_fast_zoo == 141, "if p.is_fast(): n_fast_zoo += 1  – no if on the class")
summary("2.6")

✅ 2.6 Mega is a child of Pokemon without its own __init__
✅ 2.6 Mega.describe
✅ 2.6 Mega.is_fast with 20 points less
✅ 2.6 make_any chooses the right class
✅ 2.6 per_class
✅ 2.6 n_fast_zoo == 141 (polymorphism!)

6/6 OK  🎉


## 3. Classes for your project (🧪 TEST DATA)
Now the same ideas, but for **any data table**, not only Pokémon. In part 4 these classes go into **your** `src/models.py`, so they must work with **your** columns too: **no Pokémon names inside the classes** – column names always come as parameters.

| Class | What it is | Like in part 2 |
|---|---|---|
| `Dataset` | a table: a list of rows + a name, with methods | attributes, methods, `__len__`, `__str__` |
| `Analysis` → `CountBy`, `GroupMean`, `TopGroups` | one question asked to a `Dataset` | the base class is **given and explained**, you write the children |
| `Filter` → `Equals`, `Above` | a rule which rows to keep | **you write the base class yourself** + the children |

**Why?** At the end, in `main.ipynb` you will write:
```python
for a in analyses:
    print(a.report(data))
```
One loop, many different analyses (polymorphism). A new analysis = one small new class – the loop stays the same.

`count_by` and `group_stat` used below are **the functions from your `src/stats.py`** (Lab 4) – Setup loaded them. The classes do not compute by themselves, they **use** your stats module. (If your stats.py was not ready, Setup gave you reference versions so you can practise.)

### 3.1 `Dataset` – a table as an object
**Example (a playlist – a class that keeps a list of songs):**
```python
class Playlist:
    def __init__(self, songs, name="playlist"):
        self.songs = songs                       # a list of Song objects
        self.name = name
    def __len__(self):
        return len(self.songs)                   # len(my_list) → number of songs
    def __str__(self):
        return f"Playlist '{self.name}': {len(self)} songs"
    def titles(self):
        return [s.title for s in self.songs]
    def by_artist(self, artist):                 # returns a NEW Playlist
        chosen = [s for s in self.songs if s.artist == artist]
        return Playlist(chosen, f"{self.name} | {artist}")
```
Two new ideas:
- **A method can return a new object of the same class** (`by_artist` → `Playlist`). So you can chain: `p.by_artist("Queen").titles()`.
- **Delegating:** a class can pass the work to a function that already exists (Dataset will pass its rows to `count_by` and `group_stat` from your `stats.py`). One job – one place.

**✏️ Task 3.1** – write the class `Dataset`:
| Method | Does | Expected on TEST |
|---|---|---|
| `__init__(self, rows, name="data")` | saves `rows` and `name` | |
| `__len__(self)` | number of rows | `len(data)` → `800` |
| `__str__(self)` | `"Dataset '<name>': <n> rows"` | `Dataset 'pokemon': 800 rows` |
| `column(self, name)` | list of values of one column, `None` skipped | `len(data.column("Type 2"))` → `414` |
| `where(self, column, value)` | **new** `Dataset` with rows where `row[column] == value` (name: see ① below) | `data.where("Type 1", "Water")` → 112 rows |
| `count_by(self, key)` | `count_by(self.rows, key)` from your stats.py | `data.count_by("Type 1")["Water"]` → `112` |
| `group_stat(self, group, value, how="mean")` | `group_stat(self.rows, group, value, how)` | `data.group_stat("Type 1", "Attack")["Dragon"]` → `112.12` |

① The name of the new Dataset = the old name + `" | "` + the condition: `f"{self.name} | {column} = {value}"` → `pokemon | Type 1 = Water`.

📌 **Under your nose:** `__len__` – [reference/datamodel.html → `object.__len__`](https://docs.python.org/3/reference/datamodel.html#object.__len__)

In [21]:
class Dataset:
    """A data table: a list of rows (dictionaries) + a name."""

    # Step 1: save both values inside the object:  self.rows = rows  and  self.name = name
    def __init__(self, rows, name="data"):
        self.rows = rows
        self.name = name

    # Step 2: len(data) calls this – give back the number of rows:  len(self.rows)
    def __len__(self):
        return len(self.rows)

    # Step 3: print(data) → "Dataset 'pokemon': 800 rows"   hint: f"Dataset '{self.name}': {len(self)} rows"
    def __str__(self):
        return f"Dataset '{self.name}': {len(self)} rows"

    # Step 4: a list of the values of ONE column, without None:
    #         [row[name] for row in self.rows if row[name] is not None]
    def column(self, name):
        return [row[name] for row in self.rows if row[name] is not None]

    # Step 5: keep only the rows where row[column] == value and give back a NEW Dataset:
    #         chosen = [row for row in self.rows if row[column] == value]
    #         return Dataset(chosen, f"{self.name} | {column} = {value}")
    def where(self, column, value):
        new_rows = [row for row in self.rows if row[column] == value]
        return Dataset(new_rows, f"{self.name} | {column} = {value}")

    # Step 6: do not compute yourself – call YOUR function from stats.py:  count_by(self.rows, key)
    def count_by(self, key):
        return count_by(self.rows, key)

    # Step 7: the same with group_stat:  group_stat(self.rows, group, value, how)
    def group_stat(self, group, value, how="mean"):
        return group_stat(self.rows, group, value, how)

In [22]:
# Try your Dataset on the TEST DATA
data = Dataset(TEST, "pokemon")
print(data, "| len:", len(data))
water = data.where("Type 1", "Water")
print(water)
print(water.where("Generation", 1))                 # chaining: where on the result of where
print(data.count_by("Generation"))
print(data.group_stat("Type 1", "Attack")["Dragon"])

Dataset 'pokemon': 800 rows | len: 800
Dataset 'pokemon | Type 1 = Water': 112 rows
Dataset 'pokemon | Type 1 = Water | Generation = 1': 31 rows
{1: 166, 2: 106, 3: 160, 4: 121, 5: 165, 6: 82}
112.12


In [23]:
# @title ✔️ Check 3.1
_bad = safe_test(test_dataset, Dataset)
check("3.1 Dataset works on TEST DATA", lambda: not _bad, "fix these points, run your Dataset cell again, then this check", _bad)
check("3.1 no Pokémon column names inside Dataset", lambda: Dataset([{"city": "Kraków", "t": 5}], "w").where("city", "Kraków").column("t") == [5],
      "Dataset must work for ANY columns – use the parameters column / value / key, never 'Type 1' inside the class")
summary("3.1")

✅ 3.1 Dataset works on TEST DATA
✅ 3.1 no Pokémon column names inside Dataset

2/2 OK  🎉


### 3.2 The base class `Analysis` (given – read it line by line)
An **analysis** = one question we ask a `Dataset`: "how many rows per group?", "what is the mean per group?", "which groups are the best?". Every analysis has a **name** and a way to **compute** the answer. The base class below is **given** – run it, then read the table.

In [24]:
class Analysis:
    """Base class: one analysis = one question asked to a Dataset."""

    def __init__(self, name):
        self.name = name                 # every analysis has a name (used in the report)

    def run(self, data):
        return None                      # the base class does not know HOW to compute – each child writes its own run()

    def report(self, data):
        result = self.run(data)          # calls run() of the CHILD (polymorphism!)
        return f"{self.name}: {result}"

    def __str__(self):
        return f"Analysis: {self.name}"

| Line | What it means |
|---|---|
| `class Analysis:` | a new class, no parent – it **is** the parent of all analyses |
| `def __init__(self, name):` / `self.name = name` | every analysis object remembers its name, e.g. `"count by Type 1"` |
| `def run(self, data): return None` | a **placeholder**: the base class does not know what to compute. Every child **overrides** `run` with its own calculation |
| `def report(self, data):` | written **once**, works for **all** children |
| `result = self.run(data)` | the key line: `self` is the child object (e.g. a `CountBy`), so Python finds `CountBy.run` first (child first – like in 2.5) |
| `return f"{self.name}: {result}"` | a ready line for the report: `count by Type 1: {'Grass': 70, ...}` |
| `__str__` | `print(analysis)` shows `Analysis: count by Type 1` |

**So a child must do only two things:** (1) give a name with `super().__init__("...")`, (2) write `run(self, data)`. `report` and `__str__` come for free.

**Two example children (not the ones you write):**

In [25]:
class RowCount(Analysis):
    def __init__(self):
        super().__init__("number of rows")       # (1) the name
    def run(self, data):
        return len(data)                         # (2) the calculation


class MaxOf(Analysis):
    def __init__(self, column):
        super().__init__("max " + column)        # (1) the name built from the parameter
        self.column = column                     # remember the parameter – run() needs it later
    def run(self, data):
        return max(data.column(self.column))     # (2) uses the Dataset method column()


print(RowCount().report(data))       # number of rows: 800
print(MaxOf("Attack").report(data))  # max Attack: 190
print(MaxOf("HP"))                   # Analysis: max HP   ← __str__ from Analysis

number of rows: 800
max Attack: 190
Analysis: max HP


**What happens in `MaxOf("Attack").report(data)` – step by step**
1. `MaxOf("Attack")` → `MaxOf.__init__` → `super().__init__("max Attack")` → `self.name = "max Attack"`; then `self.column = "Attack"`.
2. `.report(data)` → MaxOf has no `report` → Python takes `Analysis.report` (parent).
3. In `report`: `self.run(data)` → `self` is a MaxOf → `MaxOf.run` → `max(data.column("Attack"))` → `190`.
4. `report` returns `"max Attack: 190"`.

**✏️ Task 3.2 a** – write three children:
| Class | Parent | Name (`self.name`) | `run(data)` returns |
|---|---|---|---|
| `CountBy(column)` | `Analysis` | `"count by <column>"` | `data.count_by(column)` |
| `GroupMean(group, value)` | `Analysis` | `"mean <value> by <group>"` | `data.group_stat(group, value)` (mean) |
| `TopGroups(group, value, n=3)` | **`GroupMean`** | `"top <n> <group> by mean <value>"` | the `n` groups with the highest mean, as a list of pairs, biggest first |

`TopGroups` is a child of a child (Analysis → GroupMean → TopGroups): it uses `super().run(data)` to get the dictionary of means **from GroupMean** and only sorts it.
Sorting a dictionary by its values:
```python
means = {"Fire": 84.77, "Dragon": 112.12, "Water": 74.15}
sorted(means.items(), key=lambda pair: pair[1], reverse=True)[:2]   # [('Dragon', 112.12), ('Fire', 84.77)]
```
`means.items()` = pairs `(group, mean)`; `key=lambda pair: pair[1]` = "compare the pairs by their **second** element (the mean)"; `reverse=True` = biggest first; `[:2]` = the first 2.
📌 **Under your nose:** [howto/sorting.html → Key Functions](https://docs.python.org/3/howto/sorting.html#key-functions)

Expected: `CountBy("Type 1").run(data)["Water"]` → `112`; `TopGroups("Type 1", "Attack", 3).report(data)` → `top 3 Type 1 by mean Attack: [('Dragon', 112.12), ('Fighting', 96.78), ('Ground', 95.75)]`.

<details><summary>💡 Hint for TopGroups.__init__</summary>

`super().__init__(group, value)` – GroupMean saves group and value **and** sets the name `"mean ..."`. After it, set `self.n = n` and **overwrite** the name: `self.name = f"top {n} {group} by mean {value}"`.
</details>

In [26]:
# Every child does two things: (1) gives a name with super().__init__(...), (2) writes run(self, data).
# Look at MaxOf above – CountBy is built the same way.

class CountBy(Analysis):
    def __init__(self, column):
        # Step 1: the name:  super().__init__("count by " + column)
        # Step 2: remember the column for run():  self.column = column
        super().__init__("count by " + column)
        self.column = column

    # Step 3: the answer:  data.count_by(self.column)
    def run(self, data):
        return data.count_by(self.column)


class GroupMean(Analysis):
    def __init__(self, group, value):
        # Step 4: the name:  super().__init__(f"mean {value} by {group}")
        # Step 5: remember both:  self.group = group  and  self.value = value
        super().__init__(f"mean {value} by {group}")
        self.group = group
        self.value = value

    # Step 6: the answer:  data.group_stat(self.group, self.value)
    def run(self, data):
        return data.group_stat(self.group, self.value)


class TopGroups(GroupMean):           # a child of GroupMean, not of Analysis!
    def __init__(self, group, value, n=3):
        # Step 7: let GroupMean save group and value:  super().__init__(group, value)
        # Step 8: remember n:  self.n = n
        # Step 9: overwrite the name:  self.name = f"top {n} {group} by mean {value}"
        super().__init__(group, value)
        self.n = n
        self.name = f"top {n} {group} by mean {value}"
        
    def run(self, data):
        # Step 10: the means from GroupMean:  means = super().run(data)
        # Step 11: sort them, biggest first, keep n:
        #          return sorted(means.items(), key=lambda pair: pair[1], reverse=True)[:self.n]
        means = super().run(data)
        return sorted(means.items(), key=lambda pair: pair[1], reverse=True)[:self.n]

**✏️ Task 3.2 b** – the report loop. Make the list `analyses` with **three** analyses of your choice on TEST (one of each class, **other columns than in the examples above**, e.g. `CountBy("Generation")`), and print the report with **one loop**. Then run the same list on the subset of Water Pokémon (`data.where(...)`) – the loop does not change.

In [27]:
# Step 1: a list of three analyses – one of each class, with OTHER columns than above, e.g.
#         [CountBy("Generation"), GroupMean("Generation", "Speed"), TopGroups("Type 1", "Defense", 3)]
analyses = [CountBy("Generation"), GroupMean("Generation", "Speed"), TopGroups("Type 1", "Defense", 3)]

# Step 2: ONE loop for all of them:  print(a.report(data))
for a in analyses:
    print(a.report(data))

print("--- only Water ---")
# Step 3: only Water Pokémon – use where from your Dataset:  data.where("Type 1", "Water")
water = data.where("Type 1", "Water")
# Step 4: the same loop, but report(water)
for a in analyses:
    print(a.report(water))

count by Generation: {1: 166, 2: 106, 3: 160, 4: 121, 5: 165, 6: 82}
mean Speed by Generation: {1: 72.58, 2: 61.81, 3: 66.92, 4: 71.34, 5: 68.08, 6: 66.44}
top 3 Type 1 by mean Defense: [('Steel', 126.37), ('Rock', 100.8), ('Dragon', 86.38)]
--- only Water ---
count by Generation: {1: 31, 2: 18, 3: 27, 4: 13, 5: 18, 6: 5}
mean Speed by Generation: {1: 67.26, 2: 56.83, 3: 64.59, 4: 70.0, 5: 68.5, 6: 78.6}
top 3 Type 1 by mean Defense: [('Water', 72.95)]


In [28]:
# @title ✔️ Check 3.2
_bad = safe_test(test_analyses, types.SimpleNamespace(**{k: globals()[k] for k in ["Dataset", "Analysis", "CountBy", "GroupMean", "TopGroups"] if k in globals()}))
check("3.2 CountBy, GroupMean, TopGroups work on TEST DATA", lambda: not _bad, "fix these points, run your class cell again, then this check", _bad)
check("3.2 b analyses = 3 objects of 3 different classes", lambda: len(analyses) == 3 and {type(a).__name__ for a in analyses} == {"CountBy", "GroupMean", "TopGroups"},
      "analyses = [CountBy(...), GroupMean(...), TopGroups(...)]")
check("3.2 b other columns than in the examples", lambda: [a.name for a in analyses] != ["count by Type 1", "mean Attack by Type 1", "top 3 Type 1 by mean Attack"],
      "choose your own columns, e.g. CountBy('Generation')")
check("3.2 b water subset", lambda: len(water) == 112, 'water = data.where("Type 1", "Water")')
summary("3.2")

✅ 3.2 CountBy, GroupMean, TopGroups work on TEST DATA
✅ 3.2 b analyses = 3 objects of 3 different classes
✅ 3.2 b other columns than in the examples
✅ 3.2 b water subset

4/4 OK  🎉


### 3.3 Now YOUR base class: `Filter`
You have seen how `Analysis` is built: the **common part** (`report`) is written **once** in the base class and calls a method (`run`) that every child writes in its own way. Now you write a base class like that **yourself**.

A **filter** = a rule which rows of a `Dataset` to keep: "only Water", "only Attack > 100", "only city = Kraków", "only temperature > 30".

| `Analysis` (given) | `Filter` (you write) |
|---|---|
| `__init__(self, name)` | `__init__(self, column)` – saves `column` |
| `run(self, data)` – placeholder, children override | `keep(self, row)` – base version: `return True` (keeps every row), children override |
| `report(self, data)` – common, calls `self.run(data)` | `apply(self, data)` – common: a **new** `Dataset` with the rows for which `self.keep(row)` is `True` (name: see ① below) |
| `__str__` → `"Analysis: <name>"` | `__str__` → `"filter on <column>"` |

① `apply` → name of the result: `f"{data.name} | {self}"`, e.g. `pokemon | Attack > 100`.

Children:
| Class | `keep(row)` is `True` when | `__str__` |
|---|---|---|
| `Equals(column, value)` | `row[column] == value` | `"Type 1 = Water"` |
| `Above(column, limit)` | the value is **not None** and **bigger** than `limit` | `"Attack > 100"` |

Notice: in `apply` the name uses `{self}` – that calls `__str__` of the **child**, so the name of the result says which rule was used: `pokemon | Attack > 100`.

Expected: `Above("Attack", 100).apply(data)` → 170 rows, name `pokemon | Attack > 100`; `Equals("Type 1", "Water").apply(` that result `)` → 17 rows.

<details><summary>💡 Hint – if you do not know how to start</summary>

Look at the given `Analysis` cell and at your `Dataset.where`: `apply` is almost the same as `where`, only instead of `row[column] == value` it asks `self.keep(row)`.
`Above.keep`: `return row[self.column] is not None and row[self.column] > self.limit`
</details>

In [29]:
# Build it like Analysis above. Fill the steps in order.

class Filter:
    # Step 1: remember the column:  self.column = column
    def __init__(self, column):
        self.column = column

    # Step 2: the base filter keeps EVERY row:  return True   (children change this)
    def keep(self, row):
        return True

    # Step 3: like where() in your Dataset, but ask self.keep(row) instead of row[column] == value:
    #         kept = [row for row in data.rows if self.keep(row)]
    #         return Dataset(kept, f"{data.name} | {self}")
    def apply(self, data):
        kept = [row for row in data.rows if self.keep(row)]
        return Dataset(kept, f"{data.name} | {self}")

    # Step 4: print(Filter("HP")) → "filter on HP"
    def __str__(self):
        return f"filter on {self.column}"


class Equals(Filter):
    # Step 5: the parent saves column, you save value:  super().__init__(column)  and  self.value = value
    def __init__(self, column, value):
        super().__init__(column)
        self.value = value

    # Step 6: keep the row when  row[self.column] == self.value
    def keep(self, row):
        return row[self.column] == self.value

    # Step 7: "Type 1 = Water"   hint: f"{self.column} = {self.value}"
    def __str__(self):
        return f"{self.column} = {self.value}"


class Above(Filter):
    # Step 8: like Equals, but save limit:  super().__init__(column)  and  self.limit = limit
    def __init__(self, column, limit):
        super().__init__(column)
        self.limit = limit

    # Step 9: keep the row when the value is not None AND bigger than limit:
    #         row[self.column] is not None and row[self.column] > self.limit
    def keep(self, row):
        return row[self.column] is not None and row[self.column] > self.limit

    # Step 10: "Attack > 100"
    def __str__(self):
        return f"{self.column} > {self.limit}"

In [ ]:
# Step 1: a rule + apply it:  Above("Attack", 100).apply(data)
strong = Above("Attack", 100).apply(data)
print(strong)                                                 # expected: Dataset 'pokemon | Attack > 100': 170 rows
print(TopGroups("Type 1", "Attack", 3).report(strong))

# Step 2: a filter on the result of a filter:  Equals("Type 1", "Water").apply(strong)
strong_water = ...
print(strong_water)                                           # expected: ... 17 rows

# polymorphism again: one loop, different filters (ready – just run)
for f in [Filter("HP"), Equals("Legendary", True), Above("Speed", 120)]:
    print(f.apply(data))

In [ ]:
# @title ✔️ Check 3.3
_bad = safe_test(test_filters, types.SimpleNamespace(**{k: globals()[k] for k in ["Dataset", "Filter", "Equals", "Above"] if k in globals()}))
check("3.3 Filter, Equals, Above work on TEST DATA", lambda: not _bad, "fix these points, run your class cell again, then this check", _bad)
check("3.3 apply is written ONCE in Filter", lambda: "apply" in vars(Filter) and "apply" not in vars(Equals) and "apply" not in vars(Above),
      "apply belongs to the base class only; children write only __init__, keep, __str__")
check("3.3 strong and strong_water", lambda: (len(strong), len(strong_water)) == (170, 17) and strong_water.name == "pokemon | Attack > 100 | Type 1 = Water",
      'strong = Above("Attack", 100).apply(data);  strong_water = Equals("Type 1", "Water").apply(strong)')
summary("3.3")

## 4. Your project: `src/models.py` + `main.ipynb`
From here on you work with **your** project. If Setup showed ❌, fix the list first and run Setup again – the cells below will tell you if something is still missing.

### 4.1 Paste the classes into `src/models.py`
1. Open the file: left panel **📁 Files** → `drive` → `MyDrive` → `PyDS_project` → `src` → **double-click `models.py`**. It opens in an editor on the right (the same as `stats.py` in Lab 4).
2. Below the line `# Your code from Lab 4 goes below.` write the import line – `models.py` **uses** your `stats.py`, so it must import from it:
   ```python
   from src.stats import count_by, group_stat
   ```
3. Under it paste **from this notebook** the class cells, in this order: `Dataset` (3.1) → `Analysis` (3.2, the given one) → `CountBy`, `GroupMean`, `TopGroups` (3.2) → `Filter`, `Equals`, `Above` (3.3).
   The order matters: a child must come **after** its parent.
4. **Do not paste** the cells that use `TEST`, `data`, `print(...)` – `TEST` does not exist in your project. Do not paste `RowCount`/`MaxOf` (examples) and the Pokémon classes from part 2 (they are only practice).
5. Save: **Ctrl+S**.
6. Run the check below – it imports **your file** (the newest version) and tests the classes on the TEST DATA.

In [ ]:
# @title ✔️ Check 4.1 – my src/models.py
require_project()
try:
    MODELS = fresh("models")
    _err = None
except Exception as e:
    MODELS, _err = None, e
if _err is not None:
    _line = getattr(_err, "lineno", None)
    print(f"❌ src/models.py cannot be imported: {type(_err).__name__}: {_err}")
    if _line:
        print(f"   → open src/models.py and look at line {_line} (and the line above it).")
    if isinstance(_err, (ImportError, NameError)):
        print("   → is the line  from src.stats import count_by, group_stat  at the top of models.py?")
    RESULTS["4.1 models.py imports"] = False
else:
    check("4.1 models.py imports", lambda: True)
    check("4.1 models.py imports from src.stats", lambda: "src.stats" in (PROJECT / "src" / "models.py").read_text(encoding="utf-8"),
          "write  from src.stats import count_by, group_stat  at the top of models.py")
    for _label, _test in [("4.1 Dataset in models.py", lambda: test_dataset(MODELS.Dataset) if hasattr(MODELS, "Dataset") else ["class Dataset is missing"]),
                          ("4.1 Analysis family in models.py", lambda: test_analyses(MODELS)),
                          ("4.1 Filter family in models.py", lambda: test_filters(MODELS))]:
        try:
            _bad = _test()
        except Exception as e:
            _bad = [f"{type(e).__name__}: {e}"]
        check(_label, lambda: not _bad, "fix it in src/models.py, Ctrl+S, run this check again", _bad)
summary("4.1")

### 4.2 Rehearsal on YOUR data (here, in this notebook)
Setup loaded your data into `MY_ROWS` (with your `loader.py`) and printed your columns: **text** and **numbers**. Choose:
- `MY_GROUP` – a **text** column with **few repeating values** (a category, team, city, type …) – it makes the groups,
- `MY_VALUE` – a **number** column – it is averaged in the groups,
- one **filter** that makes sense for **your** question (e.g. `Above(MY_VALUE, <a limit>)` or `Equals(MY_GROUP, "<one value>")`).

Then write: a `Dataset` from `MY_ROWS`, a list of three analyses, the report loop, and the same report for the filtered subset. The first line loads the newest version of **your** `src/models.py`.

In [ ]:
require_project()
fresh("models")                                   # the newest version of YOUR src/models.py
from src.models import Dataset, CountBy, GroupMean, TopGroups, Equals, Above

# Step 1: two of YOUR columns – Setup printed them (text / numbers):
MY_GROUP = ...          # a TEXT column with few repeating values (category, team, city...)
MY_VALUE = ...          # a NUMBER column

# Step 2: your table:  Dataset(MY_ROWS, "my data")   (MY_ROWS = your data, loaded by Setup)
my_data = ...
print(my_data)

# Step 3: three analyses on your columns:
#         [CountBy(MY_GROUP), GroupMean(MY_GROUP, MY_VALUE), TopGroups(MY_GROUP, MY_VALUE, 3)]
my_analyses = ...

# Step 4: the report loop, like in 3.2:  print(a.report(my_data))
for a in my_analyses:
    ...

# Step 5: ONE filter that makes sense for your data, e.g.  Above(MY_VALUE, <a limit>).apply(my_data)
#         choose the limit so that SOME rows stay (not 0 and not all)
my_subset = ...
print(my_subset)

# Step 6: one analysis on the subset:  print(TopGroups(MY_GROUP, MY_VALUE, 3).report(my_subset))
...

In [ ]:
# @title ✔️ Check 4.2
require_project()
check("4.2 MY_GROUP and MY_VALUE are your columns", lambda: MY_GROUP in MY_TYPES and MY_VALUE in MY_TYPES,
      f"choose from your columns: {list(MY_TYPES)}")
check("4.2 MY_VALUE is a number column", lambda: MY_TYPES[MY_VALUE] in (int, float), "MY_VALUE must have type int or float in COLUMN_TYPES")
check("4.2 my_data = Dataset of your rows", lambda: type(my_data).__module__ == "src.models" and len(my_data) == len(MY_ROWS),
      'my_data = Dataset(MY_ROWS, "my data")  – Dataset imported from src.models')
check("4.2 three analyses that give results", lambda: len(my_analyses) >= 3 and all(a.run(my_data) not in (None, {}, []) for a in my_analyses),
      "my_analyses = [CountBy(MY_GROUP), GroupMean(MY_GROUP, MY_VALUE), TopGroups(MY_GROUP, MY_VALUE, 3)]")
check("4.2 the filtered subset is smaller and not empty", lambda: 0 < len(my_subset) < len(my_data),
      "choose a limit / value that keeps SOME rows: not 0 and not all")
summary("4.2")

### 4.3 Write it into `main.ipynb`
1. Open `main.ipynb` **properly** (like in Lab 4): **File → Open notebook → Google Drive** → `PyDS_project/main.ipynb`. (A double-click in the 📁 panel only shows a text preview – you cannot run it there.)
2. Find the section **`## Lab 4 – Classes (src/models.py)`** and its cell `# Lab 4 – your code here`. (The numbers in main come from the old plan – classes are there, the name does not matter.)
3. Write the code there **yourself**, the same steps as in 4.2, but **adapted to main**:
   - in main your data is in the variable **`rows`** (from the Lab 2 section), not `MY_ROWS`,
   - in main there is **no** `fresh(...)` and no `require_project()` – the Setup cell of main already loads the newest `src/`,
   - write your column names directly (main has no `MY_GROUP`), or define them in main first.
4. At the end add one comment line: `# Finding: ...` – what the classes showed about your data.
5. **Runtime → Run all** in main. Every cell must run without errors.
6. Come back here and run the check below. It runs your `main.ipynb` from top to bottom (it does not change it).

In [ ]:
# @title ✔️ Check 4.3 – my main.ipynb
require_project()
_code = main_code()
check("4.3 main.ipynb imports from src.models", lambda: "src.models" in _code,
      "in the section 'Lab 4 – Classes' write:  from src.models import Dataset, CountBy, ...")
MAIN_OK, _msg, MAIN_NS = run_main()
check("4.3 main.ipynb runs from top to bottom", lambda: MAIN_OK, f"error in {_msg} → fix it in main.ipynb and Run all there")
check("4.3 main.ipynb creates objects of your classes",
      lambda: any(type(v).__module__ == "src.models" for v in MAIN_NS.values()),
      "e.g.  data = Dataset(rows, '...')  and  analyses = [CountBy(...), ...]")
check("4.3 main.ipynb has a # Finding: line", lambda: "finding:" in _code.lower(), "add a comment line  # Finding: ...")
summary("4.3")

### 4.4 Finding and two explanations (in your own words)
Write full sentences – the teacher reads them. No copying of the text from this notebook.

In [ ]:
FINDING = "..."
# what did the classes show about YOUR data? (the same as the # Finding line in main)

EXPLAIN_SELF = "..."
# what is self – explain it on YOUR Dataset

EXPLAIN_INHERITANCE = "..."
# why does TopGroups inherit from GroupMean, and how can one loop run different analyses?

## 5. Result check
Run the cell below at the end. It checks everything once more and saves a short report **`docs/lab05_check.md`** in your project (this is the **only** cell of this notebook that writes a file). Then push to GitHub, if your team uses it.

🏆 **For those who finish today:** show the teacher the **✅ 6/6**.

In [ ]:
# @title ✔️ 5. Result check – run at the end
require_project()
PART_LABELS = {
    "2": ["2.1", "2.2", "2.3", "2.4", "2.5", "2.6"],
    "3": ["3.1", "3.2", "3.3"],
}
def _part_ok(prefixes):
    missing = [p for p in prefixes if not any(l.startswith(p) for l in RESULTS)]
    bad = [l for l in RESULTS if any(l.startswith(p) for p in prefixes) and not RESULTS[l]]
    return missing, bad

def _words(text):
    if not isinstance(text, str) or text.strip() in ("", "..."):
        return 0
    return len([w for w in text.split() if w.strip(".,")])

items = []
_m, _b = _part_ok(PART_LABELS["2"])
items.append(("Part 2 – classes, methods, overloading, inheritance, polymorphism (TEST DATA)", not _m and not _b,
              f"run the check cells of: {_m}" if _m else f"fix: {_b}"))
_m, _b = _part_ok(PART_LABELS["3"])
items.append(("Part 3 – Dataset, Analysis family, Filter family (TEST DATA)", not _m and not _b,
              f"run the check cells of: {_m}" if _m else f"fix: {_b}"))
try:
    _M = fresh("models")
    _bad = (test_dataset(_M.Dataset) if hasattr(_M, "Dataset") else ["Dataset missing"]) + test_analyses(_M) + test_filters(_M)
except Exception as e:
    _bad = [f"{type(e).__name__}: {e}"]
items.append(("src/models.py – Dataset, CountBy, GroupMean, TopGroups, Filter, Equals, Above work", not _bad, "; ".join(_bad[:3]) + " (see 4.1)"))
_code = main_code()
_ok, _msg, _ns = run_main()
items.append(("main.ipynb uses src.models (objects of your classes)", "src.models" in _code and any(type(v).__module__ == "src.models" for v in _ns.values()),
              "section 'Lab 4 – Classes' in main.ipynb (see 4.3)"))
items.append(("main.ipynb runs from top to bottom", _ok, _msg))
_texts = {"FINDING": globals().get("FINDING"), "EXPLAIN_SELF": globals().get("EXPLAIN_SELF"), "EXPLAIN_INHERITANCE": globals().get("EXPLAIN_INHERITANCE")}
items.append(("Finding + two explanations (at least 8 words each)", all(_words(t) >= 8 for t in _texts.values()) and "finding:" in _code.lower(),
              "fill FINDING, EXPLAIN_SELF, EXPLAIN_INHERITANCE in 4.4 and add # Finding: in main"))

score = sum(ok for _, ok, _ in items)
print("=" * 70)
for name, ok, hint in items:
    print(("✅ " if ok else "❌ ") + name + ("" if ok else "\n     → " + hint))
print("=" * 70)
print(f"RESULT: {score}/{len(items)}" + ("  🎉 Lab 5 done!" if score == len(items) else "  → fix the ❌ and run this cell again"))

report = [f"# Lab 5 – result check ({datetime.date.today()})", "", f"**Result: {score}/{len(items)}**", ""]
report += [f"- {'[OK]' if ok else '[X]'} {name}" for name, ok, _ in items]
report += ["", "## Finding", str(_texts["FINDING"]), "", "## What is self", str(_texts["EXPLAIN_SELF"]),
           "", "## Inheritance and polymorphism", str(_texts["EXPLAIN_INHERITANCE"]), ""]
try:
    (PROJECT / "docs").mkdir(exist_ok=True)
    (PROJECT / "docs" / "lab05_check.md").write_text("\n".join(report), encoding="utf-8")
    print("📝 saved: docs/lab05_check.md")
except Exception as e:
    print("Could not save the report:", e)

## What you have after Lab 5
- `src/models.py` – **your** classes: `Dataset`, the `Analysis` family and the `Filter` family. They use `src/stats.py` (Lab 4), which uses the data from `src/loader.py` (Lab 2): `loader → stats → models → main`.
- `main.ipynb` – a new section that builds a `Dataset` from your rows and runs a list of analyses with one loop.
- `docs/lab05_check.md` – your result.

**Words to remember:** class · object · `__init__` · `self` · attribute (of the object / of the class) · method · default parameter · special methods `__str__`, `__eq__`, `__lt__`, `__len__` · inheritance · `super()` · overriding · polymorphism.

### Sources
- Python tutorial, chapter 9 "Classes": https://docs.python.org/3/tutorial/classes.html
- Python reference, "Data model" – special method names: https://docs.python.org/3/reference/datamodel.html#special-method-names
- Built-in functions `super()` and `isinstance()`: https://docs.python.org/3/library/functions.html
- Sorting HOW TO (key functions): https://docs.python.org/3/howto/sorting.html
- Default argument values: https://docs.python.org/3/tutorial/controlflow.html#default-argument-values
- 🧪 TEST DATA – Pokémon dataset (`pokemon_data.csv`) from the repository KeithGalli/pandas: https://github.com/KeithGalli/pandas